<p style="text-align:center">
    <a href="https://skills.network/?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMDeveloperSkillsNetworkST0151ENSkillsNetwork20531532-2022-01-01" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>



# **Final Project: Boston Housing**


##### Estimated time needed: **60** minutes


#### Import the required libraries we need for the lab.


## Load the Boston Housing dataset

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
from statsmodels.formula.api import ols
from IPython.display import display

DATA_URL = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-ST0151EN-SkillsNetwork/labs/boston_housing.csv"
boston_df = pd.read_csv(DATA_URL)
print("Dataset shape:", boston_df.shape)
print("Columns:", boston_df.columns.tolist())

## Task 1: Familiarize yourself with the dataset

### Question 1 — First five rows

In [ ]:
display(boston_df.head())

### Question 2 — Column data types

In [ ]:
print(boston_df.dtypes)

### Question 3 — Missing values by column

In [ ]:
print(boston_df.isnull().sum())

### Question 4 — Duplicate rows

In [ ]:
print("Number of duplicate rows:", boston_df.duplicated().sum())

### Question 5 — Correlation matrix

In [ ]:
display(boston_df.corr(numeric_only=True).round(3))

## Task 2: Descriptive statistics and visualizations

### Question 6 — Descriptive statistics

In [ ]:
display(boston_df.describe(include="all"))

### Question 7 — Boxplot of MEDV

In [ ]:
plt.figure(figsize=(7, 5))
sns.boxplot(y=boston_df['MEDV'])
plt.title('Boxplot of Median Value of Owner-Occupied Homes')
plt.ylabel('MEDV ($1,000s)')
plt.show()
print('Interpretation: The plot displays the median, interquartile range, whiskers, and potential high-price outliers.')

### Question 8 — Bar plot of CHAS

In [ ]:
plt.figure(figsize=(7, 4))
sns.countplot(data=boston_df, x='CHAS', order=[0, 1])
plt.title('Homes by Charles River Boundary (CHAS)')
plt.xlabel('CHAS (0 = not bounded, 1 = bounded)')
plt.ylabel('Number of observations')
plt.show()
print('Interpretation: Compare the heights of the bars to see how many tracts are bounded by the river.')

### Question 9 — AGE categories and MEDV boxplot

In [ ]:
# AGE represents the percentage of owner-occupied units built prior to 1940.
# The requested categories describe this percentage, not homeowners' actual ages.
boston_df['AGE_group'] = pd.cut(boston_df['AGE'], bins=[-np.inf, 35, 70, np.inf], labels=['35 or younger', '36 to 70', '71 or older'])
plt.figure(figsize=(9, 5))
sns.boxplot(data=boston_df, x='AGE_group', y='MEDV', order=['35 or younger','36 to 70','71 or older'])
plt.title('MEDV by AGE Group')
plt.xlabel('AGE group (% units built before 1940)')
plt.ylabel('MEDV ($1,000s)')
plt.show()
print('Median MEDV by group:')
print(boston_df.groupby('AGE_group', observed=True)['MEDV'].median())

### Question 10 — Scatterplot of NOX vs. INDUS

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=boston_df, x='INDUS', y='NOX', alpha=0.65)
plt.title('NOX vs INDUS')
plt.xlabel('INDUS: proportion of non-retail business acres')
plt.ylabel('NOX: nitric oxide concentration')
plt.show()
print('Interpretation: An upward pattern indicates higher NOX concentrations in areas with more non-retail business acreage.')

### Question 11 — Histogram of PTRATIO

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(data=boston_df, x='PTRATIO', bins=15, kde=True)
plt.title('Distribution of Pupil–Teacher Ratio')
plt.xlabel('PTRATIO')
plt.ylabel('Frequency')
plt.show()
print('Interpretation: The histogram reveals the most frequent pupil–teacher ratios and the distribution shape.')

## Task 3: Statistical tests (significance level α = 0.05)

In [ ]:
alpha = 0.05
river = boston_df.loc[boston_df["CHAS"] == 1, "MEDV"].dropna()
nonriver = boston_df.loc[boston_df["CHAS"] == 0, "MEDV"].dropna()

### Question 12 — Levene’s test for equality of variances

In [ ]:
# H0: The two population variances of MEDV are equal.
# H1: The two population variances of MEDV differ.
lev_stat, lev_p = stats.levene(river, nonriver, center='mean')
print(f"Levene statistic = {lev_stat:.4f}; p-value = {lev_p:.6g}")
print('Reject H0: variances differ.' if lev_p < alpha else 'Fail to reject H0: insufficient evidence of unequal variances.')

### Question 13 — Independent two-sample t-test

In [ ]:
# H0: The mean MEDV is the same for CHAS = 1 and CHAS = 0.
# H1: The two means differ.
# Welch's test is used if Levene's test indicates unequal variances.
t_stat, t_p = stats.ttest_ind(river, nonriver, equal_var=(lev_p >= alpha))
print(f"t-statistic = {t_stat:.4f}; p-value = {t_p:.6g}")
print('Reject H0: the mean values differ significantly.' if t_p < alpha else 'Fail to reject H0: no significant difference established.')

### Question 14 — One-way ANOVA across AGE groups

In [ ]:
# H0: All three population mean MEDV values are equal.
# H1: At least one group has a different population mean.
groups = [boston_df.loc[boston_df['AGE_group'] == g, 'MEDV'].dropna() for g in ['35 or younger','36 to 70','71 or older']]
f_stat, anova_p = stats.f_oneway(*groups)
print(f"ANOVA F = {f_stat:.4f}; p-value = {anova_p:.6g}")
print('Reject H0: at least one group mean differs.' if anova_p < alpha else 'Fail to reject H0: no significant group mean difference established.')

### Question 15 — Pearson correlation between NOX and INDUS

In [ ]:
# H0: Population Pearson correlation is zero.
# H1: Population Pearson correlation is non-zero.
valid = boston_df[['NOX','INDUS']].dropna()
r, corr_p = stats.pearsonr(valid['NOX'], valid['INDUS'])
print(f"Pearson r = {r:.4f}; p-value = {corr_p:.6g}")
print('Reject H0: significant linear correlation.' if corr_p < alpha else 'Fail to reject H0: no significant linear correlation established.')
print('Direction:', 'positive' if r > 0 else 'negative' if r < 0 else 'zero')

### Question 16 — Simple linear regression: MEDV ~ DIS

In [ ]:
# H0: The slope of DIS equals zero.
# H1: The slope of DIS differs from zero.
reg_data = boston_df[['MEDV', 'DIS']].dropna()
model = ols('MEDV ~ DIS', data=reg_data).fit()
print(model.summary())
slope = model.params['DIS']
p = model.pvalues['DIS']
print(f"Interpretation: A one-unit increase in DIS is associated with an estimated {slope:+.3f} thousand dollars change in MEDV, on average.")
print(f"R-squared = {model.rsquared:.3f}; slope p-value = {p:.6g}.")
print('The slope is statistically significant at 5%.' if p < alpha else 'The slope is not statistically significant at 5%.')
plt.figure(figsize=(8,5))
sns.regplot(data=reg_data, x='DIS', y='MEDV', scatter_kws={'alpha':0.4})
plt.title('Linear Regression: MEDV vs DIS')
plt.xlabel('DIS: weighted distance to employment centres')
plt.ylabel('MEDV ($1,000s)')
plt.show()

## Conclusion
This notebook addresses all 16 requested exercises. Interpretations are printed beside the statistical results. The dataset is historical and the results describe associations, not causal effects.